# NB08a v3 (with NB12 OOF predictions schema) -- Temporal Analysis & Classifier Comparison

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.



| Field | Value |
|---|---|
| Notebook | `08a_BDA_Temporal_Analysis_v2.ipynb` |
| Version | `v2` |
| Pipeline position | After NB07 (Dietrich), before NB09 (ML experiments). |
| Purpose | Temporal tier x sensor ablation, classifier comparison, temporal profiles. |
| v2 changes | Uses v2 parquets via manifest. One parquet per cell, gc.collect after each. |

# CONFIG + GLOBAL SETUP

In [1]:
# @title CELL 1: NB08a v2 CONFIG + GLOBAL SETUP
TIER_SELECTION = [0,1,2]
CITY_SELECTION = None
TARGET_COL = 'damage_binary'
RANDOM_STATE = 42
FILTER_UNOSAT_ONLY = True
N_FOLDS = 5

import platform, os
if platform.system() == "Windows":
    _setup = r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py"
elif os.path.exists("/content/drive_f"):
    _setup = "/content/drive_f/masterthesis/notebooks/global_setup.py"
else:
    _setup = "/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py"
with open(_setup) as f:
    exec(f.read())

import re

# Central column-role filter: single source of truth for id / label / metadata
# classification. Replaces the hand-maintained EXCLUDE_PATTERNS list + _NON_FEATURE
# set that previously lived in this notebook. The pattern rules in metadata_filter
# catch was_observed_*, scenes_observed_*, qa__cloud_freq__*, visibility__*__freq__*
# (cloud/fire/smoke/clear), obs_count__*, and block/rolling __count_* columns --
# all of which the old EXCLUDE_PATTERNS (4 entries) + _NON_FEATURE set did NOT
# fully cover.
import importlib
import metadata_filter
importlib.reload(metadata_filter)
from metadata_filter import is_non_feature

def exclude_leakage(feat_cols):
    # Kept as a thin wrapper so existing experiment cells (C7/C9/C15) don't break.
    # Semantics: return only columns that are not id/label/metadata per the
    # central filter. Strictly MORE restrictive than the old 4-pattern filter.
    return [c for c in feat_cols if not is_non_feature(c)]


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


BDA GLOBAL SETUP
Started: 2026-04-23 22:14:39
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------
  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: False
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     910.1/7452.0 GB (6542.0 GB free)
  GDrive (F:)     1209.4/3726.0 GB (2516.6 GB free)
  Local data      11612.4/14901.9 GB (3289.5 GB free)
  Data stack      1209.4/3726.0 GB (2516.6 GB free)
  WSL ext4        68.6/1006.9 GB (887.1 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

# CELL S0: PARQUET LOADER + FEATURE CONSTRUCTION

In [2]:
# @title CELL S0: LOAD v2 MANIFEST + BUILDINGS + HELPERS
import sys, importlib, gc, re, time
import numpy as np
import pandas as pd
import json as _json
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.impute import SimpleImputer
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore")

print("=" * 70)
print("CELL S0: NB08a v2 MANIFEST + BUILDINGS + HELPERS")
print("=" * 70)

DATASET_ROOT_V2 = STACK_DIR / 'dataset' / 'v2'
V2_DIR = DATASET_ROOT_V2
MANIFEST_PATH = V2_DIR / 'parquet_manifest.json'

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"v2 manifest not found: {MANIFEST_PATH}")

with open(MANIFEST_PATH) as f:
    MANIFEST = _json.load(f)
print(f"  Manifest: {len(MANIFEST['parquets'])} parquets")

_tiers = TIER_SELECTION if TIER_SELECTION != "ALL" else [0, 1, 2, 3, 4, 5]

def load_v2_parquet(name):
    dfs = []
    for tier in _tiers:
        p = V2_DIR / f"bda_{name}_t{tier}.parquet"
        if p.exists():
            dfs.append(pd.read_parquet(p))
    if not dfs:
        return None
    return pd.concat(dfs, ignore_index=True)

df_buildings = load_v2_parquet('buildings')
if df_buildings is None:
    raise FileNotFoundError("bda_buildings not found")
if FILTER_UNOSAT_ONLY:
    df_buildings = df_buildings[df_buildings[TARGET_COL].isin([0, 1])].reset_index(drop=True)
print(f"  Buildings: {len(df_buildings)} rows, {df_buildings['city'].nunique()} cities")
print(f"  Damaged: {(df_buildings[TARGET_COL]==1).sum()}, Undamaged: {(df_buildings[TARGET_COL]==0).sum()}")

# _NON_FEATURE set removed -- metadata_filter.is_non_feature() (imported in
# CELL 1 CONFIG) is the single source of truth. Verified to cover all 43
# previously enumerated items plus pattern-matched leakage columns.

join_cols = ['building_id', 'city']

def get_analysis_df(pq_name):
    df_pq = load_v2_parquet(pq_name)
    if df_pq is None:
        return None, [], False
    is_w = 'date' not in df_pq.columns
    if is_w:
        bex = [c for c in df_buildings.columns if c not in df_pq.columns]
        merged = df_pq.merge(df_buildings[join_cols + bex], on=join_cols, how='left')
    else:
        bex = [c for c in df_buildings.columns if c not in df_pq.columns and c not in ('date', 'timestep', 'period_label')]
        merged = df_pq.merge(df_buildings[join_cols + bex], on=join_cols, how='left')
    del df_pq
    if FILTER_UNOSAT_ONLY and TARGET_COL in merged.columns:
        merged = merged[merged[TARGET_COL].isin([0, 1])].reset_index(drop=True)
    if CITY_SELECTION is not None:
        merged = merged[merged['city'].isin(CITY_SELECTION)].reset_index(drop=True)
    feat = [c for c in merged.columns
            if not is_non_feature(c)
            and merged[c].dtype.kind in ('f', 'i', 'u')]
    mb = merged.memory_usage(deep=True).sum() / 1e6
    print(f"  Loaded {pq_name}: {len(merged)} rows, {len(feat)} features, {mb:.1f} MB")
    return merged, feat, is_w

CITIES_TO_PROCESS = sorted(df_buildings['city'].unique())

RF_PARAMS = {'n_estimators': 200, 'min_samples_leaf': 3, 'random_state': RANDOM_STATE,
             'n_jobs': -1, 'class_weight': 'balanced'}

def run_experiment(df_exp, feat_cols, clf=None, use_groupkfold=True):
    if clf is None:
        clf = RandomForestClassifier(**RF_PARAMS)
    feat_clean = [c for c in feat_cols if c in df_exp.columns]
    nan_mask = df_exp[feat_clean].isna().all()
    feat_clean = [c for c in feat_clean if not nan_mask[c]]
    if len(feat_clean) < 2:
        return None
    y = df_exp[TARGET_COL].values
    groups = df_exp['city'].values
    if len(np.unique(y)) < 2:
        return None
    imp = SimpleImputer(strategy='median')
    X = imp.fit_transform(df_exp[feat_clean].values)
    n_folds = min(N_FOLDS, len(np.unique(groups)))
    if use_groupkfold and n_folds >= 2:
        cv = GroupKFold(n_splits=n_folds)
        splits = list(cv.split(X, y, groups))
    else:
        cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        splits = list(cv.split(X, y))
    y_proba = np.full(len(y), np.nan)
    fold_id = np.full(len(y), -1, dtype=int)
    for k, (train_idx, test_idx) in enumerate(splits):
        clf_copy = clone(clf)
        clf_copy.fit(X[train_idx], y[train_idx])
        if hasattr(clf_copy, 'predict_proba'):
            y_proba[test_idx] = clf_copy.predict_proba(X[test_idx])[:, 1]
        else:
            raw = clf_copy.decision_function(X[test_idx])
            y_proba[test_idx] = 1.0 / (1.0 + np.exp(-raw))
        fold_id[test_idx] = k
    valid = ~np.isnan(y_proba)
    auc = roc_auc_score(y[valid], y_proba[valid])
    f1 = f1_score(y[valid], (y_proba[valid] >= 0.5).astype(int), zero_division=0)
    return {'auc': auc, 'f1': f1, 'n_features': len(feat_clean),
            'n_buildings': int(valid.sum()), 'n_cities': len(np.unique(groups)),
            'y_proba': y_proba, 'fold_id': fold_id}

import matplotlib.pyplot as plt
OUT_DIR = RESULTS_ROOT / 'nb08'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name, subdir=''):
    d = OUT_DIR / subdir if subdir else OUT_DIR
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

def save_result(df_result, name, subdir=''):
    d = OUT_DIR / subdir if subdir else OUT_DIR
    d.mkdir(parents=True, exist_ok=True)
    out = d / f'{name}.csv'
    df_result.to_csv(out, index=False)
    print(f"    saved -> {out.name}")

NB08_RESULTS = []

# ---- NB12 overlap-analysis: canonical OOF predictions schema ----
import hashlib as _hashlib, datetime as _dt
EXPERIMENT_ID = _dt.datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + _hashlib.md5(
    f'nb08a_{TIER_SELECTION}_{FILTER_UNOSAT_ONLY}'.encode()).hexdigest()[:6]
print(f"  EXPERIMENT_ID: {EXPERIMENT_ID}")

OOF_DIR = OUT_DIR / 'oof_predictions'
OOF_DIR.mkdir(parents=True, exist_ok=True)

def save_oof(res, model_id, building_ids, cities, y_true,
             variant_id='', is_final=False, threshold=0.5):
    if res is None or 'y_proba' not in res or 'fold_id' not in res:
        print(f"    [save_oof] skipped {model_id}: missing y_proba or fold_id")
        return None
    y_proba = np.asarray(res['y_proba'])
    fold = np.asarray(res['fold_id']).astype(int)
    yt = np.asarray(y_true).astype(int)
    bid = np.asarray(building_ids)
    cty = np.asarray(cities)
    valid = ~np.isnan(y_proba)
    n_dropped = int((~valid).sum())
    if n_dropped > 0:
        print(f"    [save_oof] {model_id}: dropping {n_dropped} rows with NaN y_proba")
    y_proba = y_proba[valid]; fold = fold[valid]; yt = yt[valid]
    bid = bid[valid]; cty = cty[valid]
    yp = (y_proba >= threshold).astype(int)
    cm_class = np.where(yt == 1,
                        np.where(yp == 1, 'TP', 'FN'),
                        np.where(yp == 1, 'FP', 'TN'))
    oof_df = pd.DataFrame({
        'building_id':   bid,
        'city':          cty,
        'fold_id':       fold,
        'y_true':        yt,
        'y_proba':       y_proba.astype(float),
        'y_pred':        yp,
        'cm_class':      cm_class,
        'model_id':      model_id,
        'experiment_id': EXPERIMENT_ID,
        'variant_id':    variant_id,
        'is_final':      bool(is_final),
    })
    out = OOF_DIR / f'oof_{model_id}__{EXPERIMENT_ID}.parquet'
    oof_df.to_parquet(out, index=False)
    n_tp = int((cm_class == 'TP').sum()); n_fn = int((cm_class == 'FN').sum())
    n_fp = int((cm_class == 'FP').sum()); n_tn = int((cm_class == 'TN').sum())
    print(f"    saved oof -> {out.name}  TP={n_tp} FN={n_fn} FP={n_fp} TN={n_tn}")
    return out



CELL S0: NB08a v2 MANIFEST + BUILDINGS + HELPERS
  Manifest: 31 parquets
  Buildings: 598595 rows, 21 cities
  Damaged: 7332, Undamaged: 591263
  EXPERIMENT_ID: 20260423_221659_5eca93


In [3]:
# @title CELL S0b: OOF PLOT + SUMMARY HELPERS
# TP=red (destroyed, correctly predicted)
# TN=green (not destroyed, correctly predicted)
# FP=pink (predicted damaged but undamaged)
# FN=gold (actually damaged but missed)

CM_COLORS = {'TP': '#d62728', 'TN': '#2ca02c', 'FP': '#ff9ecb', 'FN': '#ffd700'}
CM_LABELS = {'TP': 'Destroyed (TP)', 'TN': 'Not destroyed (TN)',
             'FP': 'False positive (FP)', 'FN': 'False negative (FN)'}

def print_cm_summary(oof_path_or_df):
    if isinstance(oof_path_or_df, (str, Path)):
        oof = pd.read_parquet(oof_path_or_df)
    else:
        oof = oof_path_or_df
    model_id = oof['model_id'].iloc[0] if 'model_id' in oof.columns else '(unknown)'
    print(f"  model_id: {model_id}   n={len(oof)}")
    overall = oof['cm_class'].value_counts()
    for cls in ['TP', 'TN', 'FP', 'FN']:
        n = int(overall.get(cls, 0))
        pct = 100.0 * n / len(oof) if len(oof) else 0
        print(f"    {cls:3s} {CM_LABELS[cls]:28s}  n={n:6d}  ({pct:5.1f}%)")
    by_city = (oof.groupby('city')['cm_class']
                 .value_counts().unstack(fill_value=0))
    for cls in ['TP', 'TN', 'FP', 'FN']:
        if cls not in by_city.columns:
            by_city[cls] = 0
    by_city = by_city[['TP', 'TN', 'FP', 'FN']]
    by_city['recall']    = by_city['TP'] / (by_city['TP'] + by_city['FN']).replace(0, np.nan)
    by_city['precision'] = by_city['TP'] / (by_city['TP'] + by_city['FP']).replace(0, np.nan)
    print(f"\n  Per-city:")
    print(by_city.to_string(float_format=lambda x: f'{x:.3f}' if pd.notna(x) else '-'))
    return by_city

def plot_cm_spatial(oof_path_or_df, save_name=None, figsize=(14, 10),
                    buildings_df=None, point_size=4):
    if isinstance(oof_path_or_df, (str, Path)):
        oof = pd.read_parquet(oof_path_or_df)
    else:
        oof = oof_path_or_df
    bdf = buildings_df if buildings_df is not None else df_buildings
    if 'centroid_x' not in bdf.columns or 'centroid_y' not in bdf.columns:
        print("  plot_cm_spatial: no centroid_x/centroid_y in buildings_df")
        return None
    plot_df = oof.merge(bdf[['building_id', 'city', 'centroid_x', 'centroid_y']],
                        on=['building_id', 'city'], how='left')
    plot_df = plot_df.dropna(subset=['centroid_x', 'centroid_y'])
    cities = sorted(plot_df['city'].unique())
    if not cities:
        return None
    ncols = min(3, len(cities))
    nrows = (len(cities) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    model_id = oof['model_id'].iloc[0] if 'model_id' in oof.columns else ''
    fig.suptitle(f'Confusion-matrix map: {model_id}', fontsize=11)
    # fixed draw order so reds/golds sit on top of greens/pinks
    draw_order = ['TN', 'FP', 'FN', 'TP']
    for i, city in enumerate(cities):
        ax = axes[i // ncols][i % ncols]
        cdf = plot_df[plot_df['city'] == city]
        for cls in draw_order:
            pts = cdf[cdf['cm_class'] == cls]
            if len(pts) == 0:
                continue
            ax.scatter(pts['centroid_x'], pts['centroid_y'],
                       c=CM_COLORS[cls], s=point_size, alpha=0.75,
                       edgecolors='none',
                       label=f"{CM_LABELS[cls]} (n={len(pts)})")
        ax.set_title(f"{city}  (n={len(cdf)})", fontsize=9)
        ax.set_aspect('equal')
        ax.legend(loc='best', fontsize=6, markerscale=2, framealpha=0.85)
        ax.tick_params(labelsize=7)
    for j in range(len(cities), nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    if save_name:
        save_fig(fig, save_name, 'cm_plots')
    return fig

def plot_cm_oof_all(oof_dir=None, skip_variants=()):
    d = Path(oof_dir) if oof_dir else OOF_DIR
    files = sorted(d.glob('oof_*.parquet'))
    print(f"  Found {len(files)} OOF parquets in {d}")
    for p in files:
        oof = pd.read_parquet(p)
        v = oof['variant_id'].iloc[0] if 'variant_id' in oof.columns else ''
        if v in skip_variants:
            print(f"  skip {p.name}  (variant_id={v})")
            continue
        print(f"\n  {p.name}")
        print_cm_summary(oof)
        plot_cm_spatial(oof, save_name=p.stem)


# CELL ABLATION: DATA SCENARIO x SENSOR GROUP ABLATION MATRIX

In [4]:
# @title CELL ABLATION: PARQUET x SENSOR ABLATION MATRIX (v2)
# Each parquet = one temporal scenario. No merging across parquets.
# GroupKFold across all cities.
import gc

print("=" * 70)
print("CELL ABLATION: PARQUET x SENSOR ABLATION MATRIX")
print("=" * 70)

# Parquets represent temporal scenarios:
# PREPOST: composite_prepost_bands (A9) -- period-aggregated composites
# PREPOST_SAR: prepost_single_card (A13) -- single pre/post CARD
# FULL_TEMPORAL: rolling_stats_roll7 (A17) -- rolling SAR + baselines
# FUSION: fusion_composite_cohdrop (F7) -- MS + COH drop

scenarios = [
    ('PREPOST_MS',    'composite_prepost_bands',  'A9',  'MS composites (pre+post)'),
    ('PREPOST_SAR',   'prepost_single_card',      'A13', 'SAR CARD single pre/post'),
    ('ROLLING_SAR',   'rolling_stats_roll7',      'A17', 'SAR rolling stats (window=7)'),
    ('FUSION_MS_COH', 'fusion_composite_cohdrop', 'F7',  'MS composites + COH drop'),
]

all_results = []

for scenario, pq_name, pq_id, desc in scenarios:
    print(f"\n  --- {scenario} ({pq_id}: {pq_name}) ---")
    df_pq, feat_pq, is_wide = get_analysis_df(pq_name)
    if df_pq is None:
        print(f"    SKIP: not found")
        continue
    feat_clean = exclude_leakage(feat_pq)
    nan_pct = df_pq[feat_clean].isna().mean()
    feat_clean = [c for c in feat_clean if nan_pct[c] < 1.0]

    # split into sensor groups
    card_cols = [c for c in feat_clean if c.startswith('s1__v')]
    coh_cols = [c for c in feat_clean if c.startswith('s1__coh')]
    ms_cols = [c for c in feat_clean if c.startswith('s2__')]
    all_cols = feat_clean

    sensor_sets = {
        'CARD': card_cols,
        'COH': coh_cols,
        'MS': ms_cols,
        'SAR_ALL': list(dict.fromkeys(card_cols + coh_cols)),
        'ALL': all_cols,
    }

    for sensor, cols in sensor_sets.items():
        key = f"{scenario}_{sensor}"
        if len(cols) < 2:
            all_results.append({'scenario': scenario, 'sensor': sensor, 'parquet': pq_name,
                               'auc': np.nan, 'f1': np.nan, 'n_features': len(cols)})
            continue
        result = run_experiment(df_pq, cols)
        if result is None:
            all_results.append({'scenario': scenario, 'sensor': sensor, 'parquet': pq_name,
                               'auc': np.nan, 'f1': np.nan, 'n_features': len(cols)})
            continue
        print(f"    {key:30s}: AUC={result['auc']:.3f} F1={result['f1']:.3f} nfeat={result['n_features']} n={result['n_buildings']}")
        all_results.append({'scenario': scenario, 'sensor': sensor, 'parquet': pq_name,
                           **{k: result[k] for k in ('auc', 'f1', 'n_features', 'n_buildings', 'n_cities')}})
        NB08_RESULTS.append({'cell': 'ablation', 'experiment': key, **all_results[-1]})
        save_oof(result, f'ABL_{key}',
                 df_pq['building_id'].values, df_pq['city'].values,
                 df_pq[TARGET_COL].values,
                 variant_id=f'scenario={scenario};sensor={sensor};parquet={pq_id}')

    del df_pq; gc.collect()

results_df = pd.DataFrame(all_results)
save_result(results_df, 'ablation_matrix', 'cell_ablation')

# AUC matrix
print(f"\n  AUC MATRIX:")
sensors = ['CARD', 'COH', 'MS', 'SAR_ALL', 'ALL']
print(f"  {'':20s}", end="")
for s in sensors:
    print(f"{s:>10s}", end="")
print()
for scenario, _, _, _ in scenarios:
    print(f"  {scenario:20s}", end="")
    for s in sensors:
        row = results_df[(results_df['scenario'] == scenario) & (results_df['sensor'] == s)]
        auc = row['auc'].values[0] if len(row) > 0 else np.nan
        print(f"{auc:>10.3f}" if np.isfinite(auc) else f"{'---':>10s}", end="")
    print()
print("  Memory freed")


CELL ABLATION: PARQUET x SENSOR ABLATION MATRIX

  --- PREPOST_MS (A9: composite_prepost_bands) ---
  Loaded composite_prepost_bands: 480313 rows, 135 features, 918.0 MB
    PREPOST_MS_MS                 : AUC=0.536 F1=0.013 nfeat=135 n=480313
    saved oof -> oof_ABL_PREPOST_MS_MS__20260423_221659_5eca93.parquet  TP=167 FN=7019 FP=19164 TN=453963
    PREPOST_MS_ALL                : AUC=0.536 F1=0.013 nfeat=135 n=480313
    saved oof -> oof_ABL_PREPOST_MS_ALL__20260423_221659_5eca93.parquet  TP=167 FN=7019 FP=19164 TN=453963

  --- PREPOST_SAR (A13: prepost_single_card) ---
  Loaded prepost_single_card: 598595 rows, 12 features, 287.1 MB
    PREPOST_SAR_ALL               : AUC=0.540 F1=0.029 nfeat=12 n=598595
    saved oof -> oof_ABL_PREPOST_SAR_ALL__20260423_221659_5eca93.parquet  TP=612 FN=6720 FP=33666 TN=557597

  --- ROLLING_SAR (A17: rolling_stats_roll7) ---
  Loaded rolling_stats_roll7: 598595 rows, 99 features, 784.9 MB
    ROLLING_SAR_CARD              : AUC=0.486 F1=0.029 nfe

# CELL CLASSIFIERS: TOP CLASSIFIERS x DATA SCENARIOS

In [5]:
# @title CELL CLASSIFIERS: TOP CLASSIFIERS x DATA SCENARIOS (v2)
# Tests RF, ExtraTrees, GBM, LightGBM, XGBoost on best parquets.
# GroupKFold across cities.
import gc

print("=" * 70)
print("CELL CLASSIFIERS: TOP CLASSIFIERS x DATA SCENARIOS")
print("=" * 70)

classifiers = {
    'RF_200': RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE,
                                      n_jobs=-1, class_weight='balanced'),
}
try:
    from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier
    classifiers['ExtraTrees'] = ExtraTreesClassifier(n_estimators=200, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')
    classifiers['GBM'] = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=RANDOM_STATE, subsample=0.3)

except: pass
try:
    from lightgbm import LGBMClassifier
    classifiers['LightGBM'] = LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                              class_weight='balanced', verbose=-1)
except: pass
try:
    from xgboost import XGBClassifier
    classifiers['XGBoost'] = XGBClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                            scale_pos_weight=5, eval_metric='logloss', verbosity=0)
except: pass
print(f"  Classifiers: {list(classifiers.keys())}")

# test on best parquets from ablation
test_parquets = [
    ('A9',  'composite_prepost_bands',  'MS composites'),
    ('A17', 'rolling_stats_roll7',      'SAR rolling'),
    ('F7',  'fusion_composite_cohdrop', 'MS + COH fusion'),
]

clf_results = []

for pq_id, pq_name, desc in test_parquets:
    print(f"\n  --- {pq_id}: {desc} ---")
    df_pq, feat_pq, _ = get_analysis_df(pq_name)
    if df_pq is None:
        continue
    feat_clean = exclude_leakage(feat_pq)
    nan_pct = df_pq[feat_clean].isna().mean()
    feat_clean = [c for c in feat_clean if nan_pct[c] < 1.0]

    for clf_name, clf in classifiers.items():
        result = run_experiment(df_pq, feat_clean, clf=clf)
        if result is None:
            continue
        print(f"    {clf_name:15s}: AUC={result['auc']:.3f} F1={result['f1']:.3f} nfeat={result['n_features']}")
        clf_results.append({'parquet': pq_name, 'pq_id': pq_id, 'classifier': clf_name,
                           **{k: result[k] for k in ('auc', 'f1', 'n_features', 'n_buildings', 'n_cities')}})
        NB08_RESULTS.append({'cell': 'classifiers', 'experiment': f'{pq_id}_{clf_name}',
                            'parquet': pq_name, 'auc': result['auc'], 'f1': result['f1']})
        save_oof(result, f'CLF_{pq_id}_{clf_name}',
                 df_pq['building_id'].values, df_pq['city'].values,
                 df_pq[TARGET_COL].values,
                 variant_id=f'classifier={clf_name};parquet={pq_id}')

    del df_pq; gc.collect()

if clf_results:
    clf_df = pd.DataFrame(clf_results)
    save_result(clf_df, 'classifier_comparison', 'cell_classifiers')

    # pivot table
    print(f"\n  AUC COMPARISON:")
    pivot = clf_df.pivot(index='pq_id', columns='classifier', values='auc')
    print(pivot.to_string(float_format='{:.3f}'.format))
print("  Memory freed")


CELL CLASSIFIERS: TOP CLASSIFIERS x DATA SCENARIOS
  Classifiers: ['RF_200', 'ExtraTrees', 'GBM', 'LightGBM', 'XGBoost']

  --- A9: MS composites ---
  Loaded composite_prepost_bands: 480313 rows, 135 features, 918.0 MB
    RF_200         : AUC=0.536 F1=0.013 nfeat=135
    saved oof -> oof_CLF_A9_RF_200__20260423_221659_5eca93.parquet  TP=167 FN=7019 FP=19164 TN=453963
    ExtraTrees     : AUC=0.565 F1=0.013 nfeat=135
    saved oof -> oof_CLF_A9_ExtraTrees__20260423_221659_5eca93.parquet  TP=178 FN=7008 FP=19112 TN=454015
    GBM            : AUC=0.617 F1=0.003 nfeat=135
    saved oof -> oof_CLF_A9_GBM__20260423_221659_5eca93.parquet  TP=10 FN=7176 FP=221 TN=472906
    LightGBM       : AUC=0.710 F1=0.062 nfeat=135
    saved oof -> oof_CLF_A9_LightGBM__20260423_221659_5eca93.parquet  TP=2815 FN=4371 FP=81345 TN=391782
    XGBoost        : AUC=0.627 F1=0.044 nfeat=135
    saved oof -> oof_CLF_A9_XGBoost__20260423_221659_5eca93.parquet  TP=220 FN=6966 FP=2571 TN=470556

  --- A17: SAR rol

# CELL INVENTORY: CROSS-CITY DATA INVENTORY

In [6]:
# @title CELL INVENTORY: CROSS-CITY DATA INVENTORY (v2)
# Shows which parquets have data per city (from buildings metadata).
import gc

print("=" * 70)
print("CELL INVENTORY: CROSS-CITY DATA INVENTORY")
print("=" * 70)

inv_rows = []
for city in sorted(df_buildings['city'].unique()):
    cdf = df_buildings[df_buildings['city'] == city]
    row = {
        'city': city,
        'n_bldg': len(cdf),
        'n_dam': (cdf[TARGET_COL] == 1).sum(),
        'has_card': cdf['has_card'].any() if 'has_card' in cdf.columns else False,
        'has_coh': cdf['has_coh'].any() if 'has_coh' in cdf.columns else False,
        'has_ms': cdf['has_ms'].any() if 'has_ms' in cdf.columns else False,
    }
    # determine max temporal tier
    if row['has_coh'] and row['has_card'] and row['has_ms']:
        row['max_tier'] = 'FULL'
    elif row['has_card'] and row['has_ms']:
        row['max_tier'] = 'PREPOST'
    elif row['has_ms']:
        row['max_tier'] = 'MS_ONLY'
    else:
        row['max_tier'] = 'MINIMAL'
    inv_rows.append(row)

inv_df = pd.DataFrame(inv_rows)
print(f"\n  {'City':<22s} {'Bldg':>7s} {'Dam':>5s} {'CARD':>5s} {'COH':>5s} {'MS':>5s} {'Tier'}")
print(f"  {'-'*60}")
for _, r in inv_df.iterrows():
    print(f"  {r['city']:<22s} {r['n_bldg']:>7d} {r['n_dam']:>5d}"
          f" {'yes' if r['has_card'] else 'no':>5s} {'yes' if r['has_coh'] else 'no':>5s}"
          f" {'yes' if r['has_ms'] else 'no':>5s} {r['max_tier']}")
save_result(inv_df, 'cross_city_inventory', 'cell_inventory')


CELL INVENTORY: CROSS-CITY DATA INVENTORY

  City                      Bldg   Dam  CARD   COH    MS Tier
  ------------------------------------------------------------
  Avdiivka                 12523    89   yes   yes   yes FULL
  Borodyanka                5240    62   yes   yes    no MINIMAL
  Bucha                    17215   158   yes   yes   yes FULL
  Chernihiv                51375   333   yes   yes   yes FULL
  Chornobaivka              9909     6   yes   yes   yes FULL
  Dmytrivka                13649   144   yes   yes   yes FULL
  Hostomel                 19145   511   yes   yes   yes FULL
  Irpin                    17365   249   yes   yes   yes FULL
  Kharkiv                 170104   238   yes    no   yes PREPOST
  Kherson                   9909     6   yes   yes   yes FULL
  Kramatorsk               10414    19   yes   yes   yes FULL
  Lysychansk               41600  1071   yes   yes   yes FULL
  Makariv                   5554    53   yes    no   yes PREPOST
  Mariupol       

# CELL PROFILES: TEMPORAL PROFILES FROM SCENE PARQUETS

Per-city observation density, NaN structure at source level. Explains WHY some product_prepost features are NaN.

In [7]:
# @title CELL PROFILES: TEMPORAL PROFILES FROM SCENE PARQUETS (v2)
# Reads per-date scene parquets (A2, A3) for observation counts per city per period.
import gc

print("=" * 70)
print("CELL PROFILES: TEMPORAL PROFILES FROM SCENE PARQUETS")
print("=" * 70)

for pq_name, label in [('scene_card', 'CARD'), ('scene_coh', 'COH')]:
    sdf = load_v2_parquet(pq_name)
    if sdf is None:
        print(f"\n  {label}: NOT FOUND")
        continue
    sdf = sdf[sdf['city'].isin(CITIES_TO_PROCESS)]

    date_col = 'date2' if 'date2' in sdf.columns else 'date'
    has_ts = 'timestep' in sdf.columns
    has_pl = 'period_label' in sdf.columns

    print(f"\n  {label}: {len(sdf):,} rows, {sdf['city'].nunique()} cities")
    print(f"  {'City':<22s} {'Dates':>6s} {'Bldgs':>7s}", end="")
    if has_ts:
        print(f" {'t_min':>6s} {'t_max':>6s}", end="")
    if has_pl:
        print(f" {'pre':>5s} {'cross':>5s} {'post':>5s}", end="")
    print()

    nan_risks = []
    for city in sorted(sdf['city'].unique()):
        csdf = sdf[sdf['city'] == city]
        n_dates = csdf[date_col].nunique()
        n_bldg = csdf['building_id'].nunique()
        print(f"  {city:<22s} {n_dates:>6d} {n_bldg:>7d}", end="")
        if has_ts:
            print(f" {int(csdf['timestep'].min()):>6d} {int(csdf['timestep'].max()):>6d}", end="")
        if has_pl:
            for period in ['prebattle', 'crossbattle', 'postbattle']:
                n = csdf[csdf['period_label'] == period][date_col].nunique()
                print(f" {n:>5d}", end="")
                if 0 < n < 3:
                    nan_risks.append(f"{city} {label} {period}: {n} dates")
                elif n == 0:
                    nan_risks.append(f"{city} {label} {period}: ZERO")
        print()

    if nan_risks:
        print(f"\n  NaN RISK (< 3 dates -> unreliable aggregation):")
        for r in nan_risks:
            print(f"    {r}")

    del sdf; gc.collect()
print("  Memory freed")


CELL PROFILES: TEMPORAL PROFILES FROM SCENE PARQUETS

  CARD: 19,324,905 rows, 21 cities
  City                    Dates   Bldgs  t_min  t_max   pre cross  post
  Avdiivka                   65   13360     -5     59     5    58     2
  Borodyanka                 10   49969     -5      4     5     3     2
  Bucha                       9   17367     -5      3     5     2     2
  Chernihiv                  10   58626     -5      4     5     3     2
  Chornobaivka               27   10829     -4     22     4    21     2
  Dmytrivka                   9   13650     -5      3     5     2     2
  Hostomel                   10   19798     -5      4     5     3     2
  Irpin                      10   17686     -5      4     5     3     2
  Kharkiv                    24  213845     -5     18     5    17     2
  Kherson                    28   10829     -5     22     5    21     2
  Kramatorsk                125   58518     -5    119     5   120     0
  Lysychansk                  8   47169     -5 

# CELL PRODUCTS: PRODUCT COMPARISON

Compares raw P1, rolling_stats, and block_stats features head-to-head per city.

In [8]:
# @title CELL PRODUCTS: PRODUCT COMPARISON (v2 parquets, per-city RF)
# Quick RF (50 trees, 3-fold stratified) per parquet per city.
# Compares: prepost_single_card (A13), rolling_stats (A16-A18), block_stats (A15).
import gc

print("=" * 70)
print("CELL PRODUCTS: PER-CITY PRODUCT COMPARISON")
print("=" * 70)

test_parquets = [
    ('A13', 'prepost_single_card',  'Pre/post CARD'),
    ('A16', 'rolling_stats_roll3',  'Rolling SAR (w=3)'),
    ('A17', 'rolling_stats_roll7',  'Rolling SAR (w=7)'),
    ('A18', 'rolling_stats_roll13', 'Rolling SAR (w=13)'),
    ('A15', 'block_stats',          'Block stats'),
]

auc_rows = []
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

for pq_id, pq_name, desc in test_parquets:
    print(f"\n  --- {pq_id}: {desc} ---")
    df_pq, feat_pq, _ = get_analysis_df(pq_name)
    if df_pq is None:
        continue
    feat_clean = exclude_leakage(feat_pq)
    # only SAR features for fair comparison
    sar_cols = [c for c in feat_clean if c.startswith('s1__v')]
    nan_pct = df_pq[sar_cols].isna().mean()
    sar_cols = [c for c in sar_cols if nan_pct[c] < 1.0]

    for city in sorted(df_pq['city'].unique()):
        cdf = df_pq[df_pq['city'] == city]
        avail = [c for c in sar_cols if cdf[c].notna().sum() > 20]
        if len(avail) < 2:
            continue
        y = cdf[TARGET_COL].values
        if len(np.unique(y)) < 2 or (y == 1).sum() < 10:
            continue
        imp = SimpleImputer(strategy='median')
        X = imp.fit_transform(cdf[avail].values)
        rf = RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')
        yp = np.zeros(len(y))
        for tr, te in skf.split(X, y):
            rf.fit(X[tr], y[tr])
            yp[te] = rf.predict_proba(X[te])[:, 1]
        auc = roc_auc_score(y, yp)
        auc_rows.append({'product': pq_id, 'city': city, 'auc': auc, 'n_feat': len(avail)})

    del df_pq; gc.collect()

if auc_rows:
    auc_df = pd.DataFrame(auc_rows)
    pivot = auc_df.pivot(index='city', columns='product', values='auc')
    print(f"\n  AUC per city x product:")
    print(pivot.to_string(float_format='{:.3f}'.format))

    print(f"\n  Mean AUC across cities:")
    means = auc_df.groupby('product')['auc'].agg(['mean', 'std']).sort_values('mean', ascending=False)
    for prod, row in means.iterrows():
        print(f"    {prod:<10s}: {row['mean']:.3f} (+/-{row['std']:.3f})")

    save_result(auc_df, 'product_comparison', 'cell_products')
print("  Memory freed")


CELL PRODUCTS: PER-CITY PRODUCT COMPARISON

  --- A13: Pre/post CARD ---
  Loaded prepost_single_card: 598595 rows, 12 features, 287.1 MB

  --- A16: Rolling SAR (w=3) ---
  Loaded rolling_stats_roll3: 598595 rows, 99 features, 784.9 MB

  --- A17: Rolling SAR (w=7) ---
  Loaded rolling_stats_roll7: 598595 rows, 99 features, 784.9 MB

  --- A18: Rolling SAR (w=13) ---
  Loaded rolling_stats_roll13: 598595 rows, 57 features, 555.1 MB

  --- A15: Block stats ---
  Loaded block_stats: 598595 rows, 780 features, 4520.2 MB

  AUC per city x product:
product           A15   A16   A17   A18
city                                   
Avdiivka        0.596 0.583 0.567 0.567
Borodyanka      0.452 0.449 0.449 0.449
Bucha           0.518 0.485 0.485 0.485
Chernihiv       0.445 0.438 0.438 0.438
Dmytrivka       0.519 0.477 0.477 0.477
Hostomel        0.499 0.481 0.481 0.481
Irpin           0.500 0.463 0.463 0.463
Kharkiv         0.449 0.393 0.404 0.404
Kramatorsk      0.497 0.387 0.445 0.445
Lysychans

In [9]:
# @title CELL SAVE: SAVE NB08a RESULTS
results_df = pd.DataFrame(NB08_RESULTS)
if len(results_df) > 0:
    save_result(results_df, 'nb08a_results', '')
    print(f"  {len(results_df)} experiments logged")
else:
    print("  No experiments logged")

# OOF predictions inventory (for NB12 overlap analysis)
print(f"\n  OOF predictions inventory  ({OOF_DIR})")
oof_files = sorted(OOF_DIR.glob('oof_*.parquet'))
if oof_files:
    inv_rows = []
    for p in oof_files:
        head = pd.read_parquet(p, columns=['model_id', 'variant_id', 'cm_class'])
        cm = head['cm_class'].value_counts()
        inv_rows.append({
            'file':       p.name,
            'model_id':   head['model_id'].iloc[0],
            'variant_id': head['variant_id'].iloc[0],
            'n_rows':     len(head),
            'TP':         int(cm.get('TP', 0)),
            'FN':         int(cm.get('FN', 0)),
            'FP':         int(cm.get('FP', 0)),
            'TN':         int(cm.get('TN', 0)),
        })
    inv_df = pd.DataFrame(inv_rows)
    save_result(inv_df, 'nb08a_oof_inventory', '')
    print(inv_df.to_string(index=False))
    print(f"\n  Total OOF parquets: {len(oof_files)}")
else:
    print("  No OOF parquets saved")


    saved -> nb08a_results.csv
  26 experiments logged

  OOF predictions inventory  (/content/drive_f/masterthesis/results/nb08/oof_predictions)
    saved -> nb08a_oof_inventory.csv
                                                         file                  model_id                                       variant_id  n_rows   TP   FN     FP     TN
    oof_ABL_FUSION_MS_COH_ALL__20260422_135115_5eca93.parquet     ABL_FUSION_MS_COH_ALL     scenario=FUSION_MS_COH;sensor=ALL;parquet=F7  598595  652 6680  33720 557543
    oof_ABL_FUSION_MS_COH_ALL__20260423_221659_5eca93.parquet     ABL_FUSION_MS_COH_ALL     scenario=FUSION_MS_COH;sensor=ALL;parquet=F7  480313  211 6975  18873 454254
    oof_ABL_FUSION_MS_COH_COH__20260422_135115_5eca93.parquet     ABL_FUSION_MS_COH_COH     scenario=FUSION_MS_COH;sensor=COH;parquet=F7  598595 4263 3069 143880 447383
    oof_ABL_FUSION_MS_COH_COH__20260422_144623_5eca93.parquet     ABL_FUSION_MS_COH_COH     scenario=FUSION_MS_COH;sensor=COH;parquet=F7  598